### Lendo arquivos

In [20]:
import pandas as pd

# Definindo o caminho do arquivo
arquivo = r'C:\Users\Jose Felipe\Downloads\linhas_concorrentes_tijuca.xlsx'

# Lendo as planilhas "tabela_analise" e "dados_concorrentes"
dfs = pd.read_excel(arquivo, sheet_name=['tabela_analise', 'dados_concorrentes'])

# Acessando cada DataFrame individualmente
df_tabela = dfs['tabela_analise']
df_concorrentes = dfs['dados_concorrentes']

# Exibindo as primeiras linhas de cada DataFrame
print(df_tabela.head())
print(df_concorrentes.head())


  linha_base direcao_base linha_compartilhada direcao_compartilhada  \
0        165          Ida                 117                   Ida   
1        165          Ida                 422                   Ida   
2        165          Ida               SN422                   Ida   
3        165          Ida                 583                   Ida   
4        165          Ida                 584                   Ida   

  descricao_compartilhada                   operadora_compartilhada  \
0   Central - Cosme Velho                    AUTO VIACAO ALPHA S.A.   
1    Grajaú - Cosme Velho                              TRANSURB S/A   
2    Grajaú - Cosme Velho                              TRANSURB S/A   
3    Cosme Velho - Leblon  EMPRESA DE TRANSPORTE BRASO LISBOA LTDA.   
4    Cosme Velho - Leblon  EMPRESA DE TRANSPORTE BRASO LISBOA LTDA.   

   percentual_cobertura  percentual_cobertura_2  total_pontos_linha_base  \
0                0.6667                   66.67                       

In [205]:
df_tabela.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 512 entries, 0 to 511
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   linha_base                 512 non-null    object 
 1   direcao_base               512 non-null    object 
 2   linha_compartilhada        512 non-null    object 
 3   direcao_compartilhada      512 non-null    object 
 4   descricao_compartilhada    512 non-null    object 
 5   operadora_compartilhada    512 non-null    object 
 6   percentual_cobertura       512 non-null    float64
 7   percentual_cobertura_2     512 non-null    float64
 8   total_pontos_linha_base    512 non-null    int64  
 9   num_pontos_compartilhados  512 non-null    int64  
 10  pontos_compartilhados      512 non-null    object 
dtypes: float64(2), int64(2), object(7)
memory usage: 44.1+ KB


In [206]:
df_concorrentes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13879 entries, 0 to 13878
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   data                   13879 non-null  datetime64[ns]
 1   servico_realizado      13879 non-null  object        
 2   sentido                13879 non-null  object        
 3   quantidade_viagens     13280 non-null  float64       
 4   quantidade_transacoes  13879 non-null  int64         
 5   quantidade_veiculos    13280 non-null  float64       
 6   dia_semana             13879 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 759.1+ KB


### Relatório de Transações sem viagens associadas

In [207]:
import pandas as pd
import numpy as np
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
import os

# ----- Considera que df_tabela e df_concorrentes já estão carregados no ambiente -----
# Não haverá criação ou modificação dos seus DataFrames originais.

# 1) Definição dos intervalos (exemplo: semanas de Fevereiro e Março de 2025)
intervalos = [
    ("Fevereiro - 1ª semana", pd.to_datetime("2025-02-02"), pd.to_datetime("2025-02-08")),
    ("Fevereiro - 2ª semana", pd.to_datetime("2025-02-09"), pd.to_datetime("2025-02-15")),
    ("Fevereiro - 3ª semana", pd.to_datetime("2025-02-16"), pd.to_datetime("2025-02-22")),
    ("Fevereiro - 4ª semana", pd.to_datetime("2025-02-23"), pd.to_datetime("2025-03-01")),
    ("Março - 1ª semana",    pd.to_datetime("2025-03-02"), pd.to_datetime("2025-03-08")),
    ("Março - 2ª semana",    pd.to_datetime("2025-03-09"), pd.to_datetime("2025-03-15")),
]

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

# 2) Assegura que a coluna 'data' de df_concorrentes esteja no formato datetime
df_concorrentes['data'] = pd.to_datetime(df_concorrentes['data'], dayfirst=True, errors='coerce')

# Cria a coluna 'intervalo' aplicando a função
df_concorrentes['intervalo'] = df_concorrentes['data'].apply(atribuir_intervalo)

# 3) Filtra as transações "abertas" – aquelas em que quantidade_viagens é NaN ou igual a 0.
df_transacoes_abertas = df_concorrentes[
    df_concorrentes['quantidade_viagens'].isna() | (df_concorrentes['quantidade_viagens'] == 0)
]

# 4) Define o mapeamento de direções: em df_tabela "Ida" e "Volta" correspondem a "I" e "V" em df_concorrentes.
direcao_map = {"Ida": "I", "Volta": "V"}

# 5) Cria o PDF usando ReportLab
doc = SimpleDocTemplate("relatorio_transacoes.pdf", pagesize=letter)
styles = getSampleStyleSheet()
elements = []

# Título geral do relatório
elements.append(Paragraph("Relatório de Transações Abertas sem Partidas Associadas", styles['Title']))
elements.append(Spacer(1, 12))

# 6) Agrupa df_tabela por (linha_base, direcao_base)
# Cada grupo representa, por exemplo, "165 - Ida"
grupos_base = df_tabela.groupby(['linha_base', 'direcao_base'])

# Itera sobre os grupos com índice para inserir quebra de página antes de cada novo grupo (exceto o primeiro)
for i, ((linha_base, direcao_base), grupo) in enumerate(grupos_base):
    if i > 0:
        elements.append(PageBreak())
        
    # Cabeçalho do grupo base (ex.: "165 - Ida:")
    base_header = f"{linha_base} - {direcao_base}:"
    elements.append(Paragraph(base_header, styles['Heading1']))
    elements.append(Spacer(1, 6))
    
    # Para cada intervalo definido (mantendo a ordem definida em 'intervalos')
    for (interval_label, _, _) in intervalos:
        linhas_intervalo = []  # Lista para armazenar as linhas de resultado para este intervalo
        
        # Itera sobre cada registro do grupo: cada linha representa uma relação com uma linha compartilhada
        for _, reg in grupo.iterrows():
            shared_line = reg['linha_compartilhada']
            shared_dir = reg['direcao_compartilhada']
            # Se não houver informação na direção compartilhada, usa a direção da base
            if pd.isna(shared_dir) or str(shared_dir).strip() == "":
                shared_dir = direcao_base
            
            # Converte a direção para o formato usado em df_concorrentes ("I" ou "V")
            sentido_filtrar = direcao_map.get(shared_dir, None)
            if sentido_filtrar is None:
                continue  # Pula se não houver mapeamento
            
            # Filtra df_transacoes_abertas para a linha compartilhada, sentido e intervalo atuais
            filtro = (
                (df_transacoes_abertas['servico_realizado'] == shared_line) &
                (df_transacoes_abertas['sentido'] == sentido_filtrar) &
                (df_transacoes_abertas['intervalo'] == interval_label)
            )
            df_filtrado = df_transacoes_abertas[filtro]
            
            if not df_filtrado.empty:
                total_transacoes = int(df_filtrado['quantidade_transacoes'].sum())
                # Se a direção compartilhada for diferente da base, exibe junto; caso contrário, apenas a linha
                if shared_dir != direcao_base:
                    linha_resultado = f"{shared_line} - {shared_dir}: {total_transacoes} transações"
                else:
                    linha_resultado = f"{shared_line}: {total_transacoes} transações"
                linhas_intervalo.append(linha_resultado)
        
        # Se houver resultados para este intervalo, adiciona o título do intervalo e as linhas correspondentes
        if linhas_intervalo:
            elements.append(Paragraph(f"{interval_label}:", styles['Heading2']))
            for linha in linhas_intervalo:
                elements.append(Paragraph(linha, styles['Normal']))
            elements.append(Spacer(1, 12))
    
    # Espaço extra entre grupos, caso necessário (opcional)
    elements.append(Spacer(1, 24))

# 7) Gera e salva o PDF
doc.build(elements)
print("Relatório gerado com sucesso: relatorio_transacoes.pdf")

# (Opcional) Se estiver em ambiente desktop, pode abrir o PDF automaticamente:
os.startfile("relatorio_transacoes.pdf")


Relatório gerado com sucesso: relatorio_transacoes.pdf


### Relatório_v1, considerando média de viagens e frota. Todos os dias da semana combinados.

In [208]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger

# Definição dos intervalos
intervalos = [
    ("Fevereiro - 1ª semana", pd.to_datetime("2025-02-02"), pd.to_datetime("2025-02-08")),
    ("Fevereiro - 2ª semana", pd.to_datetime("2025-02-09"), pd.to_datetime("2025-02-15")),
    ("Fevereiro - 3ª semana", pd.to_datetime("2025-02-16"), pd.to_datetime("2025-02-22")),
    ("Fevereiro - 4ª semana", pd.to_datetime("2025-02-23"), pd.to_datetime("2025-03-01")),
    ("Março - 1ª semana",    pd.to_datetime("2025-03-02"), pd.to_datetime("2025-03-08")),
    ("Março - 2ª semana",    pd.to_datetime("2025-03-09"), pd.to_datetime("2025-03-15")),
]

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': 'sum',
        'quantidade_viagens': lambda x: x.dropna().mean(),
        'quantidade_veiculos': lambda x: x.dropna().mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change() * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abreviando os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']):
            passageiros_str += f" ({int(row['passageiros_var'])}%)"
        
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']):
            partidas_str += f" ({int(row['partidas_var'])}%)"
        
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']):
            frota_str += f" ({int(row['frota_var'])}%)"
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def gerar_capa_pdf(output_dir='output', logo_path=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    print(f"PDF de capa gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        print(f"Não há linhas compartilhadas para {linha_base_str} {direcao_base}")
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> O número de \"Passageiros\" representa a soma total de passageiros durante a semana, enquanto \"Partidas\" e \"Frota\" refletem a média desses valores ao longo da semana. Os valores de \"Partidas\" e \"Frota\" foram arredondados. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Gerar tabela para a linha base na posição de referência
    titulo_base = f"Linha {linha_base} - {direcao_base}"
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Define posições para as tabelas comparativas em grid com valores específicos
    positions = [
        # Primeira linha (3 colunas)
        (margin, height - 240),                   # Esquerda
        ((width - 3*inch) / 2, height - 240),     # Centro
        (width - margin - 3*inch, height - 240),  # Direita
        
        # Segunda linha (3 colunas)
        (margin, height - 400),                   # Esquerda
        ((width - 3*inch) / 2, height - 400),     # Centro
        (width - margin - 3*inch, height - 400),  # Direita
    ]
    
    # Contador para posição atual (começando do zero para as tabelas compartilhadas)
    pos_idx = 0
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        
        # Usar percentual_cobertura_2 em vez de percentual_cobertura
        percentual = row['percentual_cobertura_2']
        
        # Verifica se há espaço para mais tabelas nesta página
        if pos_idx >= len(positions):
            # Adiciona rodapé à página atual antes de criar uma nova
            adicionar_rodape()
            
            # Salva a página atual e cria uma nova
            c.showPage()
            page_num += 1
            
            # Adiciona cabeçalho na nova página
            c.setFont("Helvetica-Bold", 14)
            c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
            
            # Tenta adicionar a logo novamente
            if logo_path:
                try:
                    c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                except Exception as e:
                    print(f"Erro ao adicionar logo na página {page_num}: {str(e)}")
            
            # Adiciona novamente a tabela base na nova página
            gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
            
            # Recomeça com a primeira posição
            pos_idx = 0
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Pega a posição atual
            posicao = positions[pos_idx]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Move para a próxima posição
            pos_idx += 1
    
    # Adiciona rodapé à última página
    adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    print(f"PDF de comparação gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    print(f"Detectadas {len(linhas_base)} combinações únicas de linhas/direções")
    print(f"Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): {linhas_base[:min(5, len(linhas_base))]}")
    if len(linhas_base) > 5:
        print(f"... e mais {len(linhas_base) - 5} combinações")
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = os.path.join(output_dir, "capa_relatorio_completo.pdf")
    
    # Cria o PDF da capa em orientação retrato
    c = canvas.Canvas(capa_pdf, pagesize=letter)
    width, height = letter
    margin = 40
    
    # Adiciona a logo
    if logo_path:
        try:
            img = Image.open(logo_path)
            img_width, img_height = img.size
            scale_factor = 1.2
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal
    c.setFont("Helvetica-Bold", 28)
    title_y = height / 2 + 80
    c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Adiciona descrição
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, f"Página {num_pagina_atual}")
    
    # Salva a capa
    c.save()
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Definir o nome do arquivo PDF temporário para esta comparação
        temp_pdf = os.path.join(output_dir, f"temp_comp_{linha_base}_{direcao_base.lower()}.pdf")
        
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes,
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                print(f"Erro ao contar páginas do PDF: {str(e)}")
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    relatorio_final = os.path.join(output_dir, "relatorio_completo_concorrencia_v1.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários
    for pdf in todos_pdfs:
        if os.path.exists(pdf) and "relatorio_completo" not in pdf:
            try:
                os.remove(pdf)
                print(f"Arquivo temporário removido: {pdf}")
            except Exception as e:
                print(f"Erro ao remover arquivo temporário {pdf}: {str(e)}")
    
    print(f"Relatório completo único gerado com sucesso: {relatorio_final}")
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Gerar relatório completo único com todas as combinações do df_tabela
    gerar_relatorio_completo_unico(
        df_tabela,
        df_concorrentes,
        logo_path=logo_path
    )

Detectadas 39 combinações únicas de linhas/direções
Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): [('165', 'Ida'), ('165', 'Volta'), ('220', 'Ida'), ('220', 'Volta'), ('229', 'Ida')]
... e mais 34 combinações
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_ida.pdf
Logo adicionada com sucesso: C:/Users/Jos

PermissionError: [Errno 13] Permission denied: 'output\\relatorio_completo_concorrencia_v1.pdf'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger

# Definição dos intervalos
intervalos = [
    ("Fevereiro - 1ª semana", pd.to_datetime("2025-02-02"), pd.to_datetime("2025-02-08")),
    ("Fevereiro - 2ª semana", pd.to_datetime("2025-02-09"), pd.to_datetime("2025-02-15")),
    ("Fevereiro - 3ª semana", pd.to_datetime("2025-02-16"), pd.to_datetime("2025-02-22")),
    ("Fevereiro - 4ª semana", pd.to_datetime("2025-02-23"), pd.to_datetime("2025-03-01")),
    ("Março - 1ª semana",    pd.to_datetime("2025-03-02"), pd.to_datetime("2025-03-08")),
    ("Março - 2ª semana",    pd.to_datetime("2025-03-09"), pd.to_datetime("2025-03-15")),
]

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': lambda x: x.dropna().mean(),
        'quantidade_viagens': lambda x: x.dropna().mean(),
        'quantidade_veiculos': lambda x: x.dropna().mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change() * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abreviando os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']):
            passageiros_str += f" ({int(row['passageiros_var'])}%)"
        
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']):
            partidas_str += f" ({int(row['partidas_var'])}%)"
        
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']):
            frota_str += f" ({int(row['frota_var'])}%)"
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def gerar_capa_pdf(output_dir='output', logo_path=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    print(f"PDF de capa gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        print(f"Não há linhas compartilhadas para {linha_base_str} {direcao_base}")
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> Os números de \"Passageiros\", \"Partidas\" e \"Frota\" representam a média diária durante a semana. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Gerar tabela para a linha base na posição de referência
    titulo_base = f"Linha {linha_base} - {direcao_base}"
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Define posições para as tabelas comparativas em grid com valores específicos
    positions = [
        # Primeira linha (3 colunas)
        (margin, height - 240),                   # Esquerda
        ((width - 3*inch) / 2, height - 240),     # Centro
        (width - margin - 3*inch, height - 240),  # Direita
        
        # Segunda linha (3 colunas)
        (margin, height - 400),                   # Esquerda
        ((width - 3*inch) / 2, height - 400),     # Centro
        (width - margin - 3*inch, height - 400),  # Direita
    ]
    
    # Contador para posição atual (começando do zero para as tabelas compartilhadas)
    pos_idx = 0
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        
        # Usar percentual_cobertura_2 em vez de percentual_cobertura
        percentual = row['percentual_cobertura_2']
        
        # Verifica se há espaço para mais tabelas nesta página
        if pos_idx >= len(positions):
            # Adiciona rodapé à página atual antes de criar uma nova
            adicionar_rodape()
            
            # Salva a página atual e cria uma nova
            c.showPage()
            page_num += 1
            
            # Adiciona cabeçalho na nova página
            c.setFont("Helvetica-Bold", 14)
            c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
            
            # Tenta adicionar a logo novamente
            if logo_path:
                try:
                    c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                except Exception as e:
                    print(f"Erro ao adicionar logo na página {page_num}: {str(e)}")
            
            # Adiciona novamente a tabela base na nova página
            gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
            
            # Recomeça com a primeira posição
            pos_idx = 0
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Pega a posição atual
            posicao = positions[pos_idx]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Move para a próxima posição
            pos_idx += 1
    
    # Adiciona rodapé à última página
    adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    print(f"PDF de comparação gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    print(f"Detectadas {len(linhas_base)} combinações únicas de linhas/direções")
    print(f"Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): {linhas_base[:min(5, len(linhas_base))]}")
    if len(linhas_base) > 5:
        print(f"... e mais {len(linhas_base) - 5} combinações")
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = os.path.join(output_dir, "capa_relatorio_completo.pdf")
    
    # Cria o PDF da capa em orientação retrato
    c = canvas.Canvas(capa_pdf, pagesize=letter)
    width, height = letter
    margin = 40
    
    # Adiciona a logo
    if logo_path:
        try:
            img = Image.open(logo_path)
            img_width, img_height = img.size
            scale_factor = 1.2
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal
    c.setFont("Helvetica-Bold", 28)
    title_y = height / 2 + 80
    c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Adiciona descrição
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, f"Página {num_pagina_atual}")
    
    # Salva a capa
    c.save()
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Definir o nome do arquivo PDF temporário para esta comparação
        temp_pdf = os.path.join(output_dir, f"temp_comp_{linha_base}_{direcao_base.lower()}.pdf")
        
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes,
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                print(f"Erro ao contar páginas do PDF: {str(e)}")
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    relatorio_final = os.path.join(output_dir, "relatorio_completo_concorrencia_teste.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários
    for pdf in todos_pdfs:
        if os.path.exists(pdf) and "relatorio_completo" not in pdf:
            try:
                os.remove(pdf)
                print(f"Arquivo temporário removido: {pdf}")
            except Exception as e:
                print(f"Erro ao remover arquivo temporário {pdf}: {str(e)}")
    
    print(f"Relatório completo único gerado com sucesso: {relatorio_final}")
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Gerar relatório completo único com todas as combinações do df_tabela
    gerar_relatorio_completo_unico(
        df_tabela,
        df_concorrentes,
        logo_path=logo_path
    )

Detectadas 39 combinações únicas de linhas/direções
Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): [('165', 'Ida'), ('165', 'Volta'), ('220', 'Ida'), ('220', 'Volta'), ('229', 'Ida')]
... e mais 34 combinações
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_ida.pdf
Logo adicionada com sucesso: C:/Users/Jos

### Relatório_v2, considerando soma de viagens e frota. Todos os dias da semana combinados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger

# Definição dos intervalos
intervalos = [
    ("Fevereiro - 1ª semana", pd.to_datetime("2025-02-02"), pd.to_datetime("2025-02-08")),
    ("Fevereiro - 2ª semana", pd.to_datetime("2025-02-09"), pd.to_datetime("2025-02-15")),
    ("Fevereiro - 3ª semana", pd.to_datetime("2025-02-16"), pd.to_datetime("2025-02-22")),
    ("Fevereiro - 4ª semana", pd.to_datetime("2025-02-23"), pd.to_datetime("2025-03-01")),
    ("Março - 1ª semana",    pd.to_datetime("2025-03-02"), pd.to_datetime("2025-03-08")),
    ("Março - 2ª semana",    pd.to_datetime("2025-03-09"), pd.to_datetime("2025-03-15")),
]

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': 'sum',
        'quantidade_viagens': 'sum',       # Modificado: agora usando sum() em vez de mean()
        'quantidade_veiculos': 'sum'       # Modificado: agora usando sum() em vez de mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change() * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abreviando os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Função segura para formatar variações percentuais 
        def formato_seguro_var(valor):
            """Formata o valor percentual de forma segura, evitando overflow"""
            if pd.isna(valor):
                return ""
            
            # Limitar valores extremos para evitar overflow
            if valor > 1e9:  # Se for maior que 1 bilhão
                return " (+∞%)"
            elif valor < -1e9:  # Se for menor que -1 bilhão
                return " (-∞%)"
            
            try:
                return f" ({int(valor)}%)"
            except (OverflowError, ValueError):
                # Em caso de overflow ou valor inválido
                if valor > 0:
                    return " (+∞%)"
                else:
                    return " (-∞%)"
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']):
            passageiros_str += formato_seguro_var(row['passageiros_var'])
        
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']):
            partidas_str += formato_seguro_var(row['partidas_var'])
        
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']):
            frota_str += formato_seguro_var(row['frota_var'])
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def gerar_capa_pdf(output_dir='output', logo_path=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    print(f"PDF de capa gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        print(f"Não há linhas compartilhadas para {linha_base_str} {direcao_base}")
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> Os números de \"Passageiros\", \"Partidas\" e \"Frota\" representam a soma total durante a semana. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Gerar tabela para a linha base na posição de referência
    titulo_base = f"Linha {linha_base} - {direcao_base}"
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Define posições para as tabelas comparativas em grid com valores específicos
    positions = [
        # Primeira linha (3 colunas)
        (margin, height - 240),                   # Esquerda
        ((width - 3*inch) / 2, height - 240),     # Centro
        (width - margin - 3*inch, height - 240),  # Direita
        
        # Segunda linha (3 colunas)
        (margin, height - 400),                   # Esquerda
        ((width - 3*inch) / 2, height - 400),     # Centro
        (width - margin - 3*inch, height - 400),  # Direita
    ]
    
    # Contador para posição atual (começando do zero para as tabelas compartilhadas)
    pos_idx = 0
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        
        # Usar percentual_cobertura_2 em vez de percentual_cobertura
        percentual = row['percentual_cobertura_2']
        
        # Verifica se há espaço para mais tabelas nesta página
        if pos_idx >= len(positions):
            # Adiciona rodapé à página atual antes de criar uma nova
            adicionar_rodape()
            
            # Salva a página atual e cria uma nova
            c.showPage()
            page_num += 1
            
            # Adiciona cabeçalho na nova página
            c.setFont("Helvetica-Bold", 14)
            c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
            
            # Tenta adicionar a logo novamente
            if logo_path:
                try:
                    c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                except Exception as e:
                    print(f"Erro ao adicionar logo na página {page_num}: {str(e)}")
            
            # Adiciona novamente a tabela base na nova página
            gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
            
            # Recomeça com a primeira posição
            pos_idx = 0
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Pega a posição atual
            posicao = positions[pos_idx]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Move para a próxima posição
            pos_idx += 1
    
    # Adiciona rodapé à última página
    adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    print(f"PDF de comparação gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    print(f"Detectadas {len(linhas_base)} combinações únicas de linhas/direções")
    print(f"Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): {linhas_base[:min(5, len(linhas_base))]}")
    if len(linhas_base) > 5:
        print(f"... e mais {len(linhas_base) - 5} combinações")
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = os.path.join(output_dir, "capa_relatorio_completo.pdf")
    
    # Cria o PDF da capa em orientação retrato
    c = canvas.Canvas(capa_pdf, pagesize=letter)
    width, height = letter
    margin = 40
    
    # Adiciona a logo
    if logo_path:
        try:
            img = Image.open(logo_path)
            img_width, img_height = img.size
            scale_factor = 1.2
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal
    c.setFont("Helvetica-Bold", 28)
    title_y = height / 2 + 80
    c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Adiciona descrição
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, f"Página {num_pagina_atual}")
    
    # Salva a capa
    c.save()
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Definir o nome do arquivo PDF temporário para esta comparação
        temp_pdf = os.path.join(output_dir, f"temp_comp_{linha_base}_{direcao_base.lower()}.pdf")
        
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes,
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                print(f"Erro ao contar páginas do PDF: {str(e)}")
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    relatorio_final = os.path.join(output_dir, "relatorio_completo_concorrencia_v2.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários
    for pdf in todos_pdfs:
        if os.path.exists(pdf) and "relatorio_completo" not in pdf:
            try:
                os.remove(pdf)
                print(f"Arquivo temporário removido: {pdf}")
            except Exception as e:
                print(f"Erro ao remover arquivo temporário {pdf}: {str(e)}")
    
    print(f"Relatório completo único gerado com sucesso: {relatorio_final}")
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Gerar relatório completo único com todas as combinações do df_tabela
    gerar_relatorio_completo_unico(
        df_tabela,
        df_concorrentes,
        logo_path=logo_path
    )

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Gerar relatório completo único com todas as combinações do df_tabela
    gerar_relatorio_completo_unico(
        df_tabela,
        df_concorrentes,
        logo_path=logo_path
    )
    

Detectadas 39 combinações únicas de linhas/direções
Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): [('165', 'Ida'), ('165', 'Volta'), ('220', 'Ida'), ('220', 'Volta'), ('229', 'Ida')]
... e mais 34 combinações
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_ida.pdf
Logo adicionada com sucesso: C:/Users/Jos

### Relatório_v3, considerando média de passageiros, viagens e frota, abertas por tipo de dia da semana.

In [222]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger

# Definição dos intervalos
intervalos = [
    ("Fevereiro - 1ª semana", pd.to_datetime("2025-02-02"), pd.to_datetime("2025-02-08")),
    ("Fevereiro - 2ª semana", pd.to_datetime("2025-02-09"), pd.to_datetime("2025-02-15")),
    ("Fevereiro - 3ª semana", pd.to_datetime("2025-02-16"), pd.to_datetime("2025-02-22")),
    ("Fevereiro - 4ª semana", pd.to_datetime("2025-02-23"), pd.to_datetime("2025-03-01")),
    ("Março - 1ª semana",    pd.to_datetime("2025-03-02"), pd.to_datetime("2025-03-08")),
    ("Março - 2ª semana",    pd.to_datetime("2025-03-09"), pd.to_datetime("2025-03-15")),
]

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': lambda x: x.dropna().mean(),
        'quantidade_viagens': lambda x: x.dropna().mean(),
        'quantidade_veiculos': lambda x: x.dropna().mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change() * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abreviando os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']):
            passageiros_str += f" ({int(row['passageiros_var'])}%)"
        
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']):
            partidas_str += f" ({int(row['partidas_var'])}%)"
        
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']):
            frota_str += f" ({int(row['frota_var'])}%)"
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def gerar_capa_pdf(output_dir='output', logo_path=None, dia_semana=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para incluir no título
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    
    # Título com o dia da semana, se fornecido
    if dia_semana:
        c.drawCentredString(width/2, title_y, f"Relatório - Concorrência ({dia_semana})")
    else:
        c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    print(f"PDF de capa gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2, dia_semana=None):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
        dia_semana: Dia da semana para filtrar os dados (opcional)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes.empty:
            print(f"Não há dados para o dia da semana: {dia_semana}")
            return None
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        print(f"Não há linhas compartilhadas para {linha_base_str} {direcao_base}")
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> Os números de \"Passageiros\", \"Partidas\" e \"Frota\" representam a média diária durante a semana. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Gerar tabela para a linha base na posição de referência
    titulo_base = f"Linha {linha_base} - {direcao_base}"
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Define posições para as tabelas comparativas em grid com valores específicos
    positions = [
        # Primeira linha (3 colunas)
        (margin, height - 240),                   # Esquerda
        ((width - 3*inch) / 2, height - 240),     # Centro
        (width - margin - 3*inch, height - 240),  # Direita
        
        # Segunda linha (3 colunas)
        (margin, height - 400),                   # Esquerda
        ((width - 3*inch) / 2, height - 400),     # Centro
        (width - margin - 3*inch, height - 400),  # Direita
    ]
    
    # Contador para posição atual (começando do zero para as tabelas compartilhadas)
    pos_idx = 0
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        
        # Usar percentual_cobertura_2 em vez de percentual_cobertura
        percentual = row['percentual_cobertura_2']
        
        # Verifica se há espaço para mais tabelas nesta página
        if pos_idx >= len(positions):
            # Adiciona rodapé à página atual antes de criar uma nova
            adicionar_rodape()
            
            # Salva a página atual e cria uma nova
            c.showPage()
            page_num += 1
            
            # Adiciona cabeçalho na nova página
            c.setFont("Helvetica-Bold", 14)
            c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
            
            # Tenta adicionar a logo novamente
            if logo_path:
                try:
                    c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                except Exception as e:
                    print(f"Erro ao adicionar logo na página {page_num}: {str(e)}")
            
            # Adiciona novamente a tabela base na nova página
            gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
            
            # Recomeça com a primeira posição
            pos_idx = 0
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Pega a posição atual
            posicao = positions[pos_idx]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Move para a próxima posição
            pos_idx += 1
    
    # Adiciona rodapé à última página
    adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    print(f"PDF de comparação gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None, dia_semana=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    Filtra os dados de concorrentes por dia_semana se fornecido.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para filtrar (opcional)
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes_filtrado = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes_filtrado.empty:
            print(f"Não há dados para o dia da semana: {dia_semana}")
            return None
    else:
        df_concorrentes_filtrado = df_concorrentes.copy()
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    print(f"Detectadas {len(linhas_base)} combinações únicas de linhas/direções")
    print(f"Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): {linhas_base[:min(5, len(linhas_base))]}")
    if len(linhas_base) > 5:
        print(f"... e mais {len(linhas_base) - 5} combinações")
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = gerar_capa_pdf(output_dir=output_dir, logo_path=logo_path, dia_semana=dia_semana)
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Definir o nome do arquivo PDF temporário para esta comparação
        temp_pdf = os.path.join(output_dir, f"temp_comp_{linha_base}_{direcao_base.lower()}.pdf")
        
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes_filtrado,  # Usar os dados filtrados por dia da semana
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual,
            dia_semana=dia_semana  # Passar o dia da semana para a função
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                print(f"Erro ao contar páginas do PDF: {str(e)}")
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    dia_semana_formatado = dia_semana.replace(" ", "_").lower() if dia_semana else ""
    relatorio_final = os.path.join(output_dir, f"relatorio_completo_concorrencia_{dia_semana_formatado}.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários
    for pdf in todos_pdfs:
        if os.path.exists(pdf) and "relatorio_completo" not in pdf:
            try:
                os.remove(pdf)
                print(f"Arquivo temporário removido: {pdf}")
            except Exception as e:
                print(f"Erro ao remover arquivo temporário {pdf}: {str(e)}")
    
    print(f"Relatório completo único gerado com sucesso: {relatorio_final}")
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Obter todos os dias da semana únicos do df_concorrentes
    dias_semana = df_concorrentes['dia_semana'].unique()
    
    # Gerar um relatório para cada dia da semana
    for dia in dias_semana:
        print(f"\n\nGerando relatório para {dia}...")
        gerar_relatorio_completo_unico(
            df_tabela,
            df_concorrentes,
            logo_path=logo_path,
            dia_semana=dia
        )



Gerando relatório para Sábado...
Detectadas 39 combinações únicas de linhas/direções
Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): [('165', 'Ida'), ('165', 'Volta'), ('220', 'Ida'), ('220', 'Volta'), ('229', 'Ida')]
... e mais 34 combinações
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de capa gerado com sucesso: output\capa_concorrencia.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_165_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png


C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png


C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_volta.pdf


...

PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_301_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_301_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/

C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100


PDF de comparação gerado com sucesso: output\comparacao_315_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png


C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_7872\2844683776.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

PDF de comparação gerado com sucesso: output\comparacao_315_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_435_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_435_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_448_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_448_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_603_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_603_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/

### Relatório V4 - Média de passageiros, frota e partidas, incluindo a 3 semana de março

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger

# Definição dos intervalos
intervalos = [
    ("Fevereiro - 2ª semana", pd.to_datetime("2025-02-09"), pd.to_datetime("2025-02-15")),
    ("Fevereiro - 3ª semana", pd.to_datetime("2025-02-16"), pd.to_datetime("2025-02-22")),
    ("Fevereiro - 4ª semana", pd.to_datetime("2025-02-23"), pd.to_datetime("2025-03-01")),
    ("Março - 1ª semana",    pd.to_datetime("2025-03-02"), pd.to_datetime("2025-03-08")),
    ("Março - 2ª semana",    pd.to_datetime("2025-03-09"), pd.to_datetime("2025-03-15")),
    ("Março - 3ª semana",    pd.to_datetime("2025-03-16"), pd.to_datetime("2025-03-22")),
]

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': lambda x: x.dropna().mean(),
        'quantidade_viagens': lambda x: x.dropna().mean(),
        'quantidade_veiculos': lambda x: x.dropna().mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change() * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abreviando os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']):
            passageiros_str += f" ({int(row['passageiros_var'])}%)"
        
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']):
            partidas_str += f" ({int(row['partidas_var'])}%)"
        
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']):
            frota_str += f" ({int(row['frota_var'])}%)"
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def gerar_capa_pdf(output_dir='output', logo_path=None, dia_semana=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para incluir no título
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    
    # Título com o dia da semana, se fornecido
    if dia_semana:
        c.drawCentredString(width/2, title_y, f"Relatório - Concorrência ({dia_semana})")
    else:
        c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    print(f"PDF de capa gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2, dia_semana=None):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
        dia_semana: Dia da semana para filtrar os dados (opcional)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes.empty:
            print(f"Não há dados para o dia da semana: {dia_semana}")
            return None
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        print(f"Não há linhas compartilhadas para {linha_base_str} {direcao_base}")
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> Os números de \"Passageiros\", \"Partidas\" e \"Frota\" representam a média diária durante a semana. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
            print(f"Logo adicionada com sucesso: {logo_path}")
        except Exception as e:
            print(f"Erro ao adicionar logo: {str(e)}")
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Gerar tabela para a linha base na posição de referência
    titulo_base = f"Linha {linha_base} - {direcao_base}"
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Define posições para as tabelas comparativas em grid com valores específicos
    positions = [
        # Primeira linha (3 colunas)
        (margin, height - 240),                   # Esquerda
        ((width - 3*inch) / 2, height - 240),     # Centro
        (width - margin - 3*inch, height - 240),  # Direita
        
        # Segunda linha (3 colunas)
        (margin, height - 400),                   # Esquerda
        ((width - 3*inch) / 2, height - 400),     # Centro
        (width - margin - 3*inch, height - 400),  # Direita
    ]
    
    # Contador para posição atual (começando do zero para as tabelas compartilhadas)
    pos_idx = 0
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        
        # Usar percentual_cobertura_2 em vez de percentual_cobertura
        percentual = row['percentual_cobertura_2']
        
        # Verifica se há espaço para mais tabelas nesta página
        if pos_idx >= len(positions):
            # Adiciona rodapé à página atual antes de criar uma nova
            adicionar_rodape()
            
            # Salva a página atual e cria uma nova
            c.showPage()
            page_num += 1
            
            # Adiciona cabeçalho na nova página
            c.setFont("Helvetica-Bold", 14)
            c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
            
            # Tenta adicionar a logo novamente
            if logo_path:
                try:
                    c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                except Exception as e:
                    print(f"Erro ao adicionar logo na página {page_num}: {str(e)}")
            
            # Adiciona novamente a tabela base na nova página
            gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
            
            # Recomeça com a primeira posição
            pos_idx = 0
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Pega a posição atual
            posicao = positions[pos_idx]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Move para a próxima posição
            pos_idx += 1
    
    # Adiciona rodapé à última página
    adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    print(f"PDF de comparação gerado com sucesso: {pdf_filename}")
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None, dia_semana=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    Filtra os dados de concorrentes por dia_semana se fornecido.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para filtrar (opcional)
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes_filtrado = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes_filtrado.empty:
            print(f"Não há dados para o dia da semana: {dia_semana}")
            return None
    else:
        df_concorrentes_filtrado = df_concorrentes.copy()
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    print(f"Detectadas {len(linhas_base)} combinações únicas de linhas/direções")
    print(f"Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): {linhas_base[:min(5, len(linhas_base))]}")
    if len(linhas_base) > 5:
        print(f"... e mais {len(linhas_base) - 5} combinações")
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = gerar_capa_pdf(output_dir=output_dir, logo_path=logo_path, dia_semana=dia_semana)
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Definir o nome do arquivo PDF temporário para esta comparação
        temp_pdf = os.path.join(output_dir, f"temp_comp_{linha_base}_{direcao_base.lower()}.pdf")
        
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes_filtrado,  # Usar os dados filtrados por dia da semana
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual,
            dia_semana=dia_semana  # Passar o dia da semana para a função
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                print(f"Erro ao contar páginas do PDF: {str(e)}")
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    dia_semana_formatado = dia_semana.replace(" ", "_").lower() if dia_semana else ""
    relatorio_final = os.path.join(output_dir, f"relatorio_completo_concorrencia_{dia_semana_formatado}_v2.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários
    for pdf in todos_pdfs:
        if os.path.exists(pdf) and "relatorio_completo" not in pdf:
            try:
                os.remove(pdf)
                print(f"Arquivo temporário removido: {pdf}")
            except Exception as e:
                print(f"Erro ao remover arquivo temporário {pdf}: {str(e)}")
    
    print(f"Relatório completo único gerado com sucesso: {relatorio_final}")
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Obter todos os dias da semana únicos do df_concorrentes
    dias_semana = df_concorrentes['dia_semana'].unique()
    
    # Gerar um relatório para cada dia da semana
    for dia in dias_semana:
        print(f"\n\nGerando relatório para {dia}...")
        gerar_relatorio_completo_unico(
            df_tabela,
            df_concorrentes,
            logo_path=logo_path,
            dia_semana=dia
        )



Gerando relatório para Sábado...
Detectadas 39 combinações únicas de linhas/direções
Primeiras 5 combinações a serem processadas (ou todas, se menos que 5): [('165', 'Ida'), ('165', 'Volta'), ('220', 'Ida'), ('220', 'Volta'), ('229', 'Ida')]
... e mais 34 combinações
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de capa gerado com sucesso: output\capa_concorrencia.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png


C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

PDF de comparação gerado com sucesso: output\comparacao_165_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png


C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf


...

PDF de comparação gerado com sucesso: output\comparacao_165_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_220_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_229_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_301_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_301_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/

C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100


PDF de comparação gerado com sucesso: output\comparacao_315_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png


C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change() * 100
C:\Users\Jose Felipe\AppData\Local\Temp\ipykernel_8384\1175354379.py:90: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

PDF de comparação gerado com sucesso: output\comparacao_315_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_435_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_435_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_448_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_448_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_603_ida.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png
PDF de comparação gerado com sucesso: output\comparacao_603_volta.pdf
Logo adicionada com sucesso: C:/Users/Jose Felipe/Downloads/

### Relatório v5, novo formato

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, Table, TableStyle
from reportlab.lib.units import inch
import os
from pathlib import Path
from PIL import Image
import datetime
from PyPDF2 import PdfMerger

# Definição dos intervalos atualizados (9 intervalos)
import pandas as pd
from datetime import datetime, timedelta
import calendar
from collections import defaultdict

from datetime import datetime, timedelta
import pandas as pd

from datetime import datetime, timedelta
import pandas as pd

def gerar_nove_intervalos_dinamicos():
    # Obter a data atual
    data_atual = datetime.now()
    
    # Encontrar o domingo da semana atual
    ajuste_domingo = data_atual.weekday() + 1  # +1 para converter de 0=segunda para 0=domingo
    if ajuste_domingo == 7:  # Se hoje for domingo
        ajuste_domingo = 0
    domingo_atual = data_atual - timedelta(days=ajuste_domingo)
    domingo_atual = domingo_atual.replace(hour=0, minute=0, second=0, microsecond=0)
    
    # Domingo da semana anterior (este deve ser o último intervalo)
    domingo_anterior = domingo_atual - timedelta(days=7)
    
    # Gerar exatamente 9 intervalos, terminando na semana anterior
    intervalos_raw = []
    for i in range(9):
        inicio = domingo_anterior - timedelta(days=7*(8-i))  # Começa 8 semanas antes da semana anterior
        fim = inicio + timedelta(days=6)
        intervalos_raw.append((inicio, fim))
    
    # Nomes dos meses em português
    nomes_meses = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho",
        7: "Julho", 8: "Agosto", 9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    
    # Agrupar intervalos por mês usando a quinta-feira como referência
    intervalos_por_mes = defaultdict(list)
    
    for inicio, fim in intervalos_raw:
        quinta_feira = inicio + timedelta(days=4)
        mes_da_quinta = quinta_feira.month
        intervalos_por_mes[mes_da_quinta].append((inicio, fim))
    
    # Numerar as semanas dentro de cada mês e criar a lista final
    intervalos_finais = []
    for mes in sorted(intervalos_por_mes.keys()):
        # Determinar quantas semanas deste mês existem no total
        todas_semanas_mes = []
        
        # Obter primeiro dia do mês
        if mes == 1:
            primeiro_dia_mes = datetime(data_atual.year, 1, 1)
        else:
            mes_anterior = mes - 1
            ano = data_atual.year
            ultimo_dia_mes_anterior = datetime(ano, mes_anterior, 1) + timedelta(days=32)
            ultimo_dia_mes_anterior = ultimo_dia_mes_anterior.replace(day=1) - timedelta(days=1)
            primeiro_dia_mes = ultimo_dia_mes_anterior + timedelta(days=1)
        
        # Encontrar domingo que inicia ou antecede o primeiro dia do mês
        ajuste = primeiro_dia_mes.weekday() + 1
        if ajuste == 7:
            ajuste = 0
        primeiro_domingo = primeiro_dia_mes - timedelta(days=ajuste)
        
        # Gerar todas as semanas do mês
        data_temp = primeiro_domingo
        while True:
            quinta = data_temp + timedelta(days=4)
            if quinta.month != mes:
                data_temp += timedelta(days=7)
                continue
            if data_temp > domingo_anterior:
                break
            todas_semanas_mes.append(data_temp)
            data_temp += timedelta(days=7)
        
        # Numerar as semanas do mês que estão em nosso intervalo
        semanas_no_intervalo = sorted(intervalos_por_mes[mes], key=lambda x: x[0])
        for inicio, fim in semanas_no_intervalo:
            # Encontrar o número desta semana no mês
            idx = 1
            for semana_inicio in todas_semanas_mes:
                if semana_inicio == inicio:
                    break
                idx += 1
            
            nome_mes = nomes_meses[mes]
            nome_intervalo = f"{nome_mes} - {idx}ª sem"
            intervalos_finais.append((nome_intervalo, pd.to_datetime(inicio), pd.to_datetime(fim)))
    
    # Print para verificar o resultado
    #print("Intervalos gerados:")
    #for nome, inicio, fim in intervalos_finais:
        #print(f"{nome}: {inicio.strftime('%d/%m/%Y')} - {fim.strftime('%d/%m/%Y')}")
    
    return intervalos_finais

# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()

# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()



# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()

def atribuir_intervalo(data):
    """
    Retorna o rótulo do intervalo no qual a data se encaixa.
    Se a data não estiver em nenhum dos intervalos, retorna "Fora de intervalo".
    """
    for rotulo, inicio, fim in intervalos:
        if inicio <= data <= fim:
            return rotulo
    return "Fora de intervalo"

def mapear_sentido(direcao):
    """
    Mapeia as direções entre os dois DataFrames.
    Converte 'Ida' -> 'I' e 'Volta' -> 'V'
    """
    mapa = {
        'Ida': 'I',
        'Volta': 'V',
        'I': 'Ida',
        'V': 'Volta'
    }
    return mapa.get(direcao, direcao)

def filtrar_dados_concorrentes(df_concorrentes, servico, sentido):
    """
    Filtra o DataFrame pelos critérios especificados e remove dados inválidos.
    """
    # Garantir que servico seja string para comparação consistente
    servico_str = str(servico).strip()
    
    df_filtrado = df_concorrentes[
        (df_concorrentes['servico_realizado'].astype(str).str.strip() == servico_str) &
        (df_concorrentes['sentido'] == sentido)
    ]
    
    # Remove registros com intervalo "Fora de intervalo"
    df_filtrado = df_filtrado[df_filtrado['intervalo'] != "Fora de intervalo"]
    df_filtrado = df_filtrado.dropna(subset=['intervalo'])
    
    return df_filtrado

def calcular_resumo(df_filtrado):
    """
    Calcula o resumo estatístico agrupado por intervalo, incluindo variação percentual.
    """
    # Ordenar os intervalos conforme a sequência definida em 'intervalos'
    ordem_intervalos = {rotulo: i for i, (rotulo, _, _) in enumerate(intervalos)}
    
    # Agrupar por intervalo
    resumo = df_filtrado.groupby('intervalo').agg({
        'quantidade_transacoes': lambda x: x.dropna().mean(),
        'quantidade_viagens': lambda x: x.dropna().mean(),
        'quantidade_veiculos': lambda x: x.dropna().mean()
    }).reset_index()
    
    # Ordenar pelos intervalos definidos
    resumo['ordem'] = resumo['intervalo'].map(ordem_intervalos)
    resumo = resumo.sort_values('ordem')
    
    # Calcular variações percentuais entre semanas consecutivas
    resumo['passageiros'] = resumo['quantidade_transacoes']
    resumo['passageiros_var'] = resumo['quantidade_transacoes'].pct_change(fill_method=None) * 100
    
    resumo['partidas'] = resumo['quantidade_viagens']
    resumo['partidas_var'] = resumo['quantidade_viagens'].pct_change(fill_method=None) * 100
    
    resumo['frota'] = resumo['quantidade_veiculos']
    resumo['frota_var'] = resumo['quantidade_veiculos'].pct_change(fill_method=None) * 100
    
    # Remover colunas de ordem e as originais
    resumo = resumo.drop(columns=['ordem', 'quantidade_transacoes', 'quantidade_viagens', 'quantidade_veiculos'])
    
    return resumo

def preparar_df_concorrentes(df_concorrentes):
    """
    Prepara o DataFrame de concorrentes adicionando a coluna de intervalo.
    """
    # Cria uma cópia para não modificar o original
    df = df_concorrentes.copy()
    
    # Assegura que a coluna 'data' esteja no formato datetime
    df['data'] = pd.to_datetime(df['data'], errors='coerce')
    
    # Cria a coluna 'intervalo' aplicando a função
    df['intervalo'] = df['data'].apply(atribuir_intervalo)
    
    return df

def gerar_tabela_compacta(canvas, titulo, dados_resumo, posicao):
    """
    Gera uma tabela compacta diretamente em um canvas existente.
    Adiciona percentuais de variação entre parênteses.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        titulo: Título da tabela
        dados_resumo: DataFrame com os dados resumidos
        posicao: Tupla (x, y) da posição na página
    """
    # Verifica se o DataFrame está vazio
    if dados_resumo.empty:
        return
        
    # Limita o número de linhas para garantir que caiba na página
    # Máximo de 10 linhas por tabela para evitar que saia da página
    if len(dados_resumo) > 10:
        dados_resumo = dados_resumo.head(10)
    
    # Prepara os dados para a tabela (sem casas decimais e abreviados)
    tabela_dados = [['Interv.', 'Pass.', 'Part.', 'Frota']]
    
    for _, row in dados_resumo.iterrows():
        # Abrevia os nomes dos intervalos para economizar espaço
        intervalo = row['intervalo']
        intervalo = intervalo.replace('semana', 'sem')
        intervalo = intervalo.replace('Janeiro', 'Jan')
        intervalo = intervalo.replace('Fevereiro', 'Fev')
        intervalo = intervalo.replace('Março', 'Mar')
        intervalo = intervalo.replace('Abril', 'Abr')
        intervalo = intervalo.replace('Maio', 'Mai')
        intervalo = intervalo.replace('Junho', 'Jun')
        intervalo = intervalo.replace('Julho', 'Jul')
        intervalo = intervalo.replace('Agosto', 'Ago')
        intervalo = intervalo.replace('Setembro', 'Set')
        intervalo = intervalo.replace('Outubro', 'Out')
        intervalo = intervalo.replace('Novembro', 'Nov')
        intervalo = intervalo.replace('Dezembro', 'Dez')
        
        # Limita o tamanho do texto do intervalo para 12 caracteres
        if len(intervalo) > 12:
            intervalo = intervalo[:9] + '...'
        
        # Prepara a formatação dos valores com variações percentuais (sem casas decimais)
        passageiros_str = f"{int(row['passageiros']):,}".replace(',', '.') if not pd.isna(row['passageiros']) else "-"
        if not pd.isna(row['passageiros_var']):
            passageiros_str += f" ({int(row['passageiros_var'])}%)"
        
        partidas_str = f"{int(row['partidas'])}" if not pd.isna(row['partidas']) else "-"
        if not pd.isna(row['partidas_var']):
            partidas_str += f" ({int(row['partidas_var'])}%)"
        
        frota_str = f"{int(row['frota'])}" if not pd.isna(row['frota']) else "-"
        if not pd.isna(row['frota_var']):
            frota_str += f" ({int(row['frota_var'])}%)"
        
        tabela_dados.append([
            intervalo,
            passageiros_str,
            partidas_str,
            frota_str
        ])
    
    # Cria uma tabela com melhor espaçamento entre colunas e coluna de intervalo reduzida
    table = Table(tabela_dados, colWidths=[0.9*inch, 1.0*inch, 0.7*inch, 0.7*inch], spaceBefore=5, spaceAfter=5)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ALIGN', (0, 1), (0, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 8),           # Fonte um pouco maior para legibilidade
        ('FONTSIZE', (0, 1), (-1, -1), 7),          # Fonte um pouco maior para legibilidade
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),     # Padding um pouco maior
        ('TOPPADDING', (0, 0), (-1, -1), 3),        # Padding um pouco maior
        ('GRID', (0, 0), (-1, -1), 1, colors.black), # Linha da grade mais grossa
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BACKGROUND', (0, 1), (-1, -1), colors.white),
    ]))
    
    # Posição da tabela
    table_x, table_y = posicao
    
    # Garante que a tabela caiba na página (ajusta posição Y se necessário)
    # Obtém as dimensões da tabela
    table_width, table_height = table.wrapOn(canvas, 300, 500)
    
    # Se a tabela for ficar fora da página, ajuste a posição Y
    if table_y - table_height < 30:  # Garante pelo menos 30 pontos de margem inferior
        table_y = 30 + table_height
    
    # Adiciona título da tabela acima dela (com mais espaço)
    canvas.setFont("Helvetica-Bold", 9)  # Fonte um pouco maior para legibilidade
    canvas.drawString(table_x, table_y + 15, titulo)  # 15 pontos acima da tabela
    
    # Desenha a tabela
    table.drawOn(canvas, table_x, table_y - table_height)

def desenhar_lista_linhas(canvas, linhas_para_desenhar, height):
    """
    Desenha uma lista com informações das linhas compartilhadas ao lado da tabela base.
    
    Args:
        canvas: Canvas do ReportLab para desenhar
        linhas_para_desenhar: Lista de tuplas (linha_comp, descricao) a serem desenhadas
        height: Altura da página para posicionamento
    """
    x = 50  # Posição X inicial
    y = height - 80  # Mesma altura da tabela base
    
    # Configura a fonte para os itens da lista
    canvas.setFont("Helvetica", 8)
    
    # Desenha os itens da lista
    for i, (linha_comp, descricao) in enumerate(linhas_para_desenhar):
        # Formato: • linha_compartilhada (descricao_compartilhada)
        texto = f"• {linha_comp}"
        if descricao and not pd.isna(descricao):
            texto += f" ({descricao})"
        
        canvas.drawString(x, y - 15 * i, texto)

def gerar_capa_pdf(output_dir='output', logo_path=None, dia_semana=None):
    """
    Função que gera uma capa em PDF para o relatório de concorrência.
    
    Args:
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para incluir no título
        
    Returns:
        str: Caminho do arquivo PDF gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"capa_concorrencia.pdf")
    
    # Cria o PDF em orientação retrato (padrão)
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    width, height = letter
    
    # Define a margem padrão
    margin = 40
    
    # Adiciona a logo no centro superior se fornecida
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 1.2  # Fator reduzido ainda mais para logo maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona a logo centralizada no topo
            logo_x = (width - logo_width) / 2
            logo_y = height - logo_height - margin
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            pass
    
    # Adiciona título principal (aumentado e posicionado mais acima)
    c.setFont("Helvetica-Bold", 28)  # Tamanho aumentado de 24 para 28
    title_y = height / 2 + 80  # Posicionado mais acima (era +50)
    
    # Título com o dia da semana, se fornecido
    if dia_semana:
        c.drawCentredString(width/2, title_y, f"Relatório - Concorrência ({dia_semana})")
    else:
        c.drawCentredString(width/2, title_y, "Relatório - Concorrência")
    
    # Linha horizontal removida conforme solicitado
    
    # Data removida conforme solicitado
    
    # Adiciona informações sobre o relatório
    info_style = ParagraphStyle(
        'Info',
        fontName='Helvetica-Oblique',
        fontSize=11,
        leading=14,
        alignment=1,  # Centralizado
    )
    
    info_text = "Análise comparativa de linhas com pontos compartilhados"
    p = Paragraph(info_text, info_style)
    p.wrapOn(c, width - 2*margin, height)
    p.drawOn(c, margin, title_y - 50)  # Ajustado para ficar mais próximo do título
    
    # Rodapé removido conforme solicitado
    
    # Adiciona número de página
    c.setFont("Helvetica", 8)
    c.drawRightString(width - margin, margin, "Página 1")
    
    # Salva o documento
    c.save()
    
    return pdf_filename

def gerar_pdf_comparacao(df_tabela, df_concorrentes, linha_base=220, direcao_base="Ida", output_dir='output', logo_path=None, pagina_inicial=2, dia_semana=None):
    """
    Função principal que gera um PDF comparando a linha base com suas linhas compartilhadas.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        linha_base: Número da linha base para análise
        direcao_base: Direção da linha base (Ida/Volta)
        output_dir: Diretório de saída para o arquivo PDF
        logo_path: Caminho para o arquivo da logo
        pagina_inicial: Número da primeira página deste relatório (default: 2, considerando a capa como página 1)
        dia_semana: Dia da semana para filtrar os dados (opcional)
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes.empty:
            return None
    
    # Mapear a direção base para o formato do df_concorrentes
    sentido_base = mapear_sentido(direcao_base)
    
    # Preparar o df_concorrentes adicionando a coluna de intervalo
    df_concorrentes_prep = preparar_df_concorrentes(df_concorrentes)
    
    # Obter todas as linhas compartilhadas para esta linha/direção base
    linha_base_str = str(linha_base).strip()
    
    # Usamos .astype(str) para converter todos os valores para string antes de comparar
    linhas_compartilhadas = df_tabela[
        (df_tabela['linha_base'].astype(str).str.strip() == linha_base_str) & 
        (df_tabela['direcao_base'].str.strip() == direcao_base.strip())
    ]
    
    # Verificar se existem linhas compartilhadas
    if linhas_compartilhadas.empty:
        return None
    
    # Define o nome do arquivo PDF
    pdf_filename = os.path.join(output_dir, f"comparacao_{linha_base}_{direcao_base.lower()}.pdf")
    
    # Cria o PDF em orientação horizontal
    c = canvas.Canvas(pdf_filename, pagesize=landscape(letter))
    width, height = landscape(letter)
    
    # Define a margem padrão
    margin = 40
    
    # Função auxiliar para adicionar rodapé à página atual
    def adicionar_rodape():
        # Calcular a posição do rodapé estendido até metade da terceira coluna
        rodape_largura = ((width - 3*inch) / 2) + (3*inch / 2) - margin  # Até a metade da terceira coluna
        
        # Criar parágrafo para o rodapé com formatação de negrito para "Nota:"
        rodape_style = ParagraphStyle(
            'Rodape',
            fontName='Helvetica-Oblique',
            fontSize=6,
            leading=8,  # Espaçamento entre linhas
        )
        
        # Usando tags HTML para negrito no texto do rodapé
        rodape_texto = "<b>Nota:</b> Os números de \"Passageiros\", \"Partidas\" e \"Frota\" representam a média diária durante a semana. Os percentuais acima da tabela indicam a cobertura compartilhada da linha em relação à linha base, enquanto os percentuais dentro da tabela mostram a variação em comparação com a semana anterior."
        
        p = Paragraph(rodape_texto, rodape_style)
        p.wrapOn(c, rodape_largura, 30)  # 30pts de altura
        p.drawOn(c, margin, 15)
        
        # Adiciona número de página
        c.setFont("Helvetica", 8)
        c.drawRightString(width - margin, 20, f"Página {page_num}")
    
    # Adiciona a logo no canto superior direito se fornecida
    logo_x = logo_y = logo_width = logo_height = 0
    if logo_path:
        try:
            # Tenta carregar a imagem com PIL para obter dimensões reais
            img = Image.open(logo_path)
            img_width, img_height = img.size
            
            # Calcula o fator de redução para manter a proporção
            scale_factor = 3  # Reduzido para logo ainda maior
            logo_width = img_width / scale_factor
            logo_height = img_height / scale_factor
            
            # Posiciona mais próximo do canto superior direito
            logo_x = width - logo_width - 20  # Reduzido o espaçamento da borda direita
            logo_y = height - logo_height + 15  # Posicionado 15pts acima
            
            # Adiciona a imagem ao PDF
            c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
        except Exception as e:
            pass
    
    # Adiciona um título principal com formato "Comparativo de Linhas (linha_base - direcao_base)"
    c.setFont("Helvetica-Bold", 14)
    c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
    
    # Filtrar dados da linha base
    df_base_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_base, sentido_base)
    
    # Calcular resumo da linha base
    resumo_base = calcular_resumo(df_base_filtrado)
    
    # Posição para a tabela de referência (centralizada no topo)
    base_x = (width - 3*inch) / 2  # Centralizado
    base_y = height - 80  # Conforme solicitado
    
    # Obter o número total de pontos da linha base
    total_pontos = None
    if not linhas_compartilhadas.empty and 'total_pontos_linha_base' in linhas_compartilhadas.columns:
        primeira_linha = linhas_compartilhadas.iloc[0]
        if 'total_pontos_linha_base' in primeira_linha:
            total_pontos = primeira_linha['total_pontos_linha_base']
    
    # Gerar tabela para a linha base na posição de referência
    if total_pontos is not None:
        titulo_base = f"Linha {linha_base} - {direcao_base} ({total_pontos} pontos)"
    else:
        titulo_base = f"Linha {linha_base} - {direcao_base}"
    
    gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Reduzindo para 3 tabelas por página (apenas uma linha de tabelas) e movendo para baixo
    positions = [
        # Apenas primeira linha (3 colunas) com posição Y mais baixa
        (margin, height - 320),                   # Esquerda
        ((width - 3*inch) / 2, height - 320),     # Centro
        (width - margin - 3*inch, height - 320),  # Direita
    ]
    
    # Variáveis para controle de página
    page_num = pagina_inicial  # Iniciar com o número de página fornecido
    tabelas_na_pagina = 0
    linhas_na_pagina_atual = []
    
    # Para cada linha compartilhada
    for idx, row in linhas_compartilhadas.iterrows():
        linha_comp = row['linha_compartilhada']
        direcao_comp = row['direcao_compartilhada']
        descricao = row.get('descricao_compartilhada', '')
        
        # Usar percentual_cobertura_2 em vez de percentual_cobertura
        percentual = row['percentual_cobertura_2']
        
        # Mapear a direção compartilhada para o formato do df_concorrentes
        if pd.isna(direcao_comp) or str(direcao_comp).strip() == "":
            direcao_comp = direcao_base
        
        sentido_comp = mapear_sentido(direcao_comp)
        
        # Filtrar dados da linha compartilhada
        df_comp_filtrado = filtrar_dados_concorrentes(df_concorrentes_prep, linha_comp, sentido_comp)
        
        # Verificar se há dados para processar
        if not df_comp_filtrado.empty:
            # Calcular resumo
            resumo_comp = calcular_resumo(df_comp_filtrado)
            
            # Formatação do percentual
            try:
                if not pd.isna(percentual):
                    percentual_float = float(percentual)
                    percentual_formatado = f"{int(percentual_float)}"
                else:
                    percentual_formatado = "N/A"
            except:
                percentual_formatado = str(percentual)
            
            # Título para esta tabela com percentual formatado (sem casas decimais)
            titulo_comp = f"Linha {linha_comp} - {direcao_comp} ({percentual_formatado}%)"
            
            # Adiciona esta linha à lista para a página atual
            linhas_na_pagina_atual.append((linha_comp, descricao))
            
            # Pega a posição atual
            posicao = positions[tabelas_na_pagina]
            
            # Cria a tabela na posição especificada
            gerar_tabela_compacta(c, titulo_comp, resumo_comp, posicao)
            
            # Incrementa contagem de tabelas
            tabelas_na_pagina += 1
            
            # Se completamos 3 tabelas, desenha a lista e cria uma nova página
            if tabelas_na_pagina == 3:
                # Desenha a lista de linhas da página atual
                desenhar_lista_linhas(c, linhas_na_pagina_atual, height)
                
                # Adiciona rodapé e prepara nova página
                adicionar_rodape()
                c.showPage()
                page_num += 1
                
                # Resetar contagens para a próxima página
                tabelas_na_pagina = 0
                linhas_na_pagina_atual = []
                
                # Adiciona cabeçalho na nova página
                c.setFont("Helvetica-Bold", 14)
                c.drawCentredString(width/2, height - 30, f"Comparativo de Linhas ({linha_base} - {direcao_base})")
                
                # Tenta adicionar a logo novamente
                if logo_path:
                    try:
                        c.drawImage(logo_path, logo_x, logo_y, width=logo_width, height=logo_height, mask='auto')
                    except Exception as e:
                        pass
                
                # Adiciona novamente a tabela base na nova página
                gerar_tabela_compacta(c, titulo_base, resumo_base, (base_x, base_y))
    
    # Se ainda temos tabelas na última página, desenha a lista de linhas e o rodapé
    if tabelas_na_pagina > 0:
        desenhar_lista_linhas(c, linhas_na_pagina_atual, height)
        adicionar_rodape()
    
    # Salva o documento
    c.save()
    
    return pdf_filename

def gerar_relatorio_completo_unico(df_tabela, df_concorrentes, output_dir='output', logo_path=None, dia_semana=None):
    """
    Gera um relatório único contendo uma capa e todos os relatórios de comparação.
    As linhas são extraídas automaticamente do df_tabela, mantendo os formatos originais.
    Filtra os dados de concorrentes por dia_semana se fornecido.
    
    Args:
        df_tabela: DataFrame com informações das linhas compartilhadas
        df_concorrentes: DataFrame com dados de concorrentes
        output_dir: Diretório de saída
        logo_path: Caminho para o arquivo da logo
        dia_semana: Dia da semana para filtrar (opcional)
        
    Returns:
        str: Caminho do relatório completo gerado
    """
    # Garantir que o diretório de saída existe
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Filtrar df_concorrentes por dia_semana se fornecido
    if dia_semana:
        df_concorrentes_filtrado = df_concorrentes[df_concorrentes['dia_semana'] == dia_semana].copy()
        if df_concorrentes_filtrado.empty:
            return None
    else:
        df_concorrentes_filtrado = df_concorrentes.copy()
    
    # Extrair todas as combinações únicas de linha_base e direcao_base
    linhas_direcoes = df_tabela[['linha_base', 'direcao_base']].drop_duplicates().reset_index(drop=True)
    
    # Criar uma coluna para ordenação dos sentidos (Ida = 1, Volta = 2, outros = 3)
    def ordem_sentido(sentido):
        if sentido == 'Ida':
            return 1
        elif sentido == 'Volta':
            return 2
        else:
            return 3
    
    linhas_direcoes['ordem_sentido'] = linhas_direcoes['direcao_base'].apply(ordem_sentido)
    
    # Como linha_base pode conter siglas, vamos manter o formato original e ordenar apenas por sentido
    linhas_direcoes = linhas_direcoes.sort_values(['linha_base', 'ordem_sentido']).reset_index(drop=True)
    
    # Converter para o formato de lista de tuplas
    linhas_base = [(str(row['linha_base']), str(row['direcao_base'])) for _, row in linhas_direcoes.iterrows()]
    
    # Lista para armazenar todos os PDFs temporários gerados
    todos_pdfs = []
    num_pagina_atual = 1
    
    # Primeiro, gerar a capa
    capa_pdf = gerar_capa_pdf(output_dir=output_dir, logo_path=logo_path, dia_semana=dia_semana)
    num_pagina_atual += 1
    todos_pdfs.append(capa_pdf)
    
    # Agora, gerar cada relatório de comparação
    for linha_base, direcao_base in linhas_base:
        # Gerar o PDF de comparação começando na página correta
        pdf_gerado = gerar_pdf_comparacao(
            df_tabela,
            df_concorrentes_filtrado,  # Usar os dados filtrados por dia da semana
            linha_base=linha_base,
            direcao_base=direcao_base,
            output_dir=output_dir,
            logo_path=logo_path,
            pagina_inicial=num_pagina_atual,
            dia_semana=dia_semana  # Passar o dia da semana para a função
        )
        
        if pdf_gerado:
            todos_pdfs.append(pdf_gerado)
            
            # Atualizar o número da próxima página inicial
            # Precisamos determinar quantas páginas foram criadas neste relatório
            try:
                import PyPDF2
                with open(pdf_gerado, 'rb') as f:
                    pdf_reader = PyPDF2.PdfReader(f)
                    num_paginas = len(pdf_reader.pages)
                    num_pagina_atual += num_paginas
            except Exception as e:
                # Supondo que cada relatório tenha ao menos 1 página
                num_pagina_atual += 1
    
    # Combinar todos os PDFs em um único documento
    dia_semana_formatado = dia_semana.replace(" ", "_").lower() if dia_semana else ""
    relatorio_final = os.path.join(output_dir, f"relatorio_completo_concorrencia_{dia_semana_formatado}.pdf")
    
    # Usar PdfMerger para mesclar os PDFs
    merger = PdfMerger()
    
    for pdf in todos_pdfs:
        if os.path.exists(pdf):
            merger.append(pdf)
    
    # Escrever o arquivo final
    merger.write(relatorio_final)
    merger.close()
    
    # Limpar arquivos temporários com força extra
    for pdf in todos_pdfs:
        try:
            if os.path.exists(pdf) and "relatorio_completo" not in pdf:
                os.remove(pdf)
        except Exception:
            try:
                # Segunda tentativa com delay
                import time
                time.sleep(0.5)
                if os.path.exists(pdf):
                    os.remove(pdf)
            except Exception:
                pass
    
    return relatorio_final

# Exemplo de uso
if __name__ == "__main__":
    # df_tabela e df_concorrentes já estão disponíveis no ambiente
    
    # Caminho para a logo
    logo_path = 'C:/Users/Jose Felipe/Downloads/Logo_Tijuca.png'
    
    # Obter todos os dias da semana únicos do df_concorrentes
    dias_semana = df_concorrentes['dia_semana'].unique()
    
    # Gerar um relatório para cada dia da semana
    for dia in dias_semana:
        gerar_relatorio_completo_unico(
            df_tabela,
            df_concorrentes,
            logo_path=logo_path,
            dia_semana=dia
        )

In [21]:
def gerar_nove_intervalos_dinamicos():
    # Obter a data atual
    data_atual = datetime.now()
    
    # Encontrar o domingo da semana atual
    ajuste_domingo = data_atual.weekday() + 1  # +1 para converter de 0=segunda para 0=domingo
    if ajuste_domingo == 7:  # Se hoje for domingo
        ajuste_domingo = 0
    domingo_atual = data_atual - timedelta(days=ajuste_domingo)
    domingo_atual = domingo_atual.replace(hour=0, minute=0, second=0, microsecond=0)
    
    # Domingo da semana anterior (este deve ser o último intervalo)
    domingo_anterior = domingo_atual - timedelta(days=7)
    
    # Gerar exatamente 9 intervalos, terminando na semana anterior
    intervalos_raw = []
    for i in range(9):
        inicio = domingo_anterior - timedelta(days=7*(8-i))  # Começa 8 semanas antes da semana anterior
        fim = inicio + timedelta(days=6)
        intervalos_raw.append((inicio, fim))
    
    # Nomes dos meses em português
    nomes_meses = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho",
        7: "Julho", 8: "Agosto", 9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    
    # Agrupar intervalos por mês usando a quinta-feira como referência
    intervalos_por_mes = defaultdict(list)
    
    for inicio, fim in intervalos_raw:
        quinta_feira = inicio + timedelta(days=4)
        mes_da_quinta = quinta_feira.month
        intervalos_por_mes[mes_da_quinta].append((inicio, fim))
    
    # Numerar as semanas dentro de cada mês e criar a lista final
    intervalos_finais = []
    for mes in sorted(intervalos_por_mes.keys()):
        # Determinar quantas semanas deste mês existem no total
        todas_semanas_mes = []
        
        # Obter primeiro dia do mês
        if mes == 1:
            primeiro_dia_mes = datetime(data_atual.year, 1, 1)
        else:
            mes_anterior = mes - 1
            ano = data_atual.year
            ultimo_dia_mes_anterior = datetime(ano, mes_anterior, 1) + timedelta(days=32)
            ultimo_dia_mes_anterior = ultimo_dia_mes_anterior.replace(day=1) - timedelta(days=1)
            primeiro_dia_mes = ultimo_dia_mes_anterior + timedelta(days=1)
        
        # Encontrar domingo que inicia ou antecede o primeiro dia do mês
        ajuste = primeiro_dia_mes.weekday() + 1
        if ajuste == 7:
            ajuste = 0
        primeiro_domingo = primeiro_dia_mes - timedelta(days=ajuste)
        
        # Gerar todas as semanas do mês
        data_temp = primeiro_domingo
        while True:
            quinta = data_temp + timedelta(days=4)
            if quinta.month != mes:
                data_temp += timedelta(days=7)
                continue
            if data_temp > domingo_anterior:
                break
            todas_semanas_mes.append(data_temp)
            data_temp += timedelta(days=7)
        
        # Numerar as semanas do mês que estão em nosso intervalo
        semanas_no_intervalo = sorted(intervalos_por_mes[mes], key=lambda x: x[0])
        for inicio, fim in semanas_no_intervalo:
            # Encontrar o número desta semana no mês
            idx = 1
            for semana_inicio in todas_semanas_mes:
                if semana_inicio == inicio:
                    break
                idx += 1
            
            nome_mes = nomes_meses[mes]
            nome_intervalo = f"{nome_mes} - {idx}ª sem"
            intervalos_finais.append((nome_intervalo, pd.to_datetime(inicio), pd.to_datetime(fim)))
    
    # Print para verificar o resultado
    print("Intervalos gerados:")
    for nome, inicio, fim in intervalos_finais:
        print(f"{nome}: {inicio.strftime('%d/%m/%Y')} - {fim.strftime('%d/%m/%Y')}")
    
    return intervalos_finais

# Gerar os intervalos dinamicamente
intervalos = gerar_nove_intervalos_dinamicos()

Intervalos gerados:
Fevereiro - 3ª sem: 16/02/2025 - 22/02/2025
Fevereiro - 4ª sem: 23/02/2025 - 01/03/2025
Março - 1ª sem: 02/03/2025 - 08/03/2025
Março - 2ª sem: 09/03/2025 - 15/03/2025
Março - 3ª sem: 16/03/2025 - 22/03/2025
Março - 4ª sem: 23/03/2025 - 29/03/2025
Abril - 1ª sem: 30/03/2025 - 05/04/2025
Abril - 2ª sem: 06/04/2025 - 12/04/2025
Abril - 3ª sem: 13/04/2025 - 19/04/2025
